# Wake Word "Letícia" para Home Assistant (v4)

## Instruções
1. GPU T4 ativa (Ambiente de execução > Alterar tipo > T4 GPU)
2. Execute célula a célula, na ordem
3. **OBRIGATÓRIO:** Após Etapa 1 → Reiniciar sessão → continuar da Etapa 1b

---

## Etapa 1a: Instalação de Dependências
⚠️ **Reiniciar a sessão após esta célula!**

In [ ]:
import os, locale
locale.getpreferredencoding = lambda *a: 'UTF-8'

print('Clonando repositórios...')
if not os.path.exists('./piper-sample-generator'):
    !git clone -q https://github.com/dscripka/piper-sample-generator
    print('[OK] piper-sample-generator')
else:
    print('[SKIP] piper-sample-generator')

if not os.path.exists('./openwakeword'):
    !git clone -q https://github.com/dscripka/openWakeWord openwakeword
    print('[OK] openWakeWord')
else:
    print('[SKIP] openWakeWord')

print('\nInstalando pacotes...')
!pip install -q pathvalidate piper-tts piper-phonemize-cross webrtcvad
!pip install -q 'torch<=2.5' torchvision torchaudio
!pip install -q -e ./openwakeword
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0
!pip install -q speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0
!pip install -q acoustics==0.2.6 scipy
!pip install -q onnxruntime ai-edge-litert onnxsim
!pip install -q onnx2tf tensorflow==2.19.0
!pip install -q onnx==1.19.1 onnx_graphsurgeon
!pip install -q datasets==2.14.6
!cd piper-sample-generator && pip install -q -r requirements.txt
print('[OK] Pacotes instalados')

print()
print('⚠️  REINICIE A SESSÃO AGORA!')
print('   Ambiente de execução > Reiniciar sessão')
print('   Depois execute as células a partir da Etapa 1b')

## Etapa 1b: Pós-reinício — Downloads iniciais
*(Execute esta célula após reiniciar a sessão)*

In [ ]:
import os, locale
import numpy as np
locale.getpreferredencoding = lambda *a: 'UTF-8'

# Teste de compatibilidade numpy
_ = np.random.RandomState(42)
print(f'numpy {np.__version__}: OK')

# Modelos de embedding do openWakeWord
models_dir = 'openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
base_url = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    fpath = os.path.join(models_dir, fname)
    if not os.path.exists(fpath):
        !wget -q '{base_url}/{fname}' -O {fpath}
        print(f'[OK] {fname}')
    else:
        print(f'[SKIP] {fname}')

# Modelo LibriTTS
libritts = 'piper-sample-generator/models/en-us-libritts-high.pt'
if not os.path.exists(libritts):
    os.makedirs('piper-sample-generator/models', exist_ok=True)
    !wget -q -O {libritts} 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
    print('[OK] LibriTTS')
else:
    print('[SKIP] LibriTTS')

# Vozes Piper pt_BR
os.makedirs('piper_voices_ptbr', exist_ok=True)
voices = {
    'pt_BR-faber-medium': 'pt/pt_BR/faber/medium',
    'pt_BR-edresson-low': 'pt/pt_BR/edresson/low',
}
hf_base = 'https://huggingface.co/rhasspy/piper-voices/resolve/main'
for name, path in voices.items():
    onnx_path = f'piper_voices_ptbr/{name}.onnx'
    if not os.path.exists(onnx_path):
        !wget -q -O {onnx_path} '{hf_base}/{path}/{name}.onnx'
        !wget -q -O {onnx_path}.json '{hf_base}/{path}/{name}.onnx.json'
        print(f'[OK] {name}')
    else:
        print(f'[SKIP] {name}')

print()
print('[OK] ETAPA 1b CONCLUÍDA!')

## Etapa 2: Testar Pronúncia
Ouça os áudios gerados para confirmar a pronúncia de "Letícia".

In [ ]:
import subprocess, os
from IPython.display import Audio, display

target_word = 'letícia'
os.makedirs('test_audio', exist_ok=True)

for name, label in [('pt_BR-faber-medium','Faber'), ('pt_BR-edresson-low','Edresson')]:
    out = f'test_audio/test_{label.lower()}.wav'
    r = subprocess.run(['piper','--model',f'piper_voices_ptbr/{name}.onnx','--output_file',out],
                       input=target_word, capture_output=True, text=True)
    if os.path.exists(out):
        print(f'Voz {label} (pt_BR):')
        display(Audio(out, autoplay=False))
    else:
        print(f'[AVISO] {label}: {r.stderr[:200]}')

print('[OK] ETAPA 2 CONCLUÍDA!')

## Etapa 3: Download de Dados Auxiliares

Baixa dados exatamente como o notebook oficial do openWakeWord:
- **RIRs**: `davidscripka/MIT_environmental_impulse_responses` (split=`train`)
- **AudioSet**: tar de `agkphysics/AudioSet` → converte para 16kHz WAV
- **FMA música**: `rudraml/fma` (name=`small`, split=`train`)
- **ACAV100M features**: wget direto do HuggingFace (arquivo .npy)
- **Validation features**: wget direto do HuggingFace (arquivo .npy)

**Tempo estimado: ~15-30 minutos**

In [ ]:
import os, numpy as np, scipy.io.wavfile as wav
import datasets
from tqdm import tqdm

print('=' * 60)
print('  ETAPA 3: Baixando dados auxiliares...')
print('=' * 60)

# ============================================================
# 3a. Room Impulse Responses (MIT)
# Dataset correto: davidscripka/MIT_environmental_impulse_responses
# ============================================================
if not os.path.exists('mit_rirs') or len([f for f in os.listdir('mit_rirs') if f.endswith('.wav')]) == 0:
    print('\n[3a] Baixando MIT Room Impulse Responses...')
    os.makedirs('mit_rirs', exist_ok=True)
    rir_dataset = datasets.load_dataset(
        'davidscripka/MIT_environmental_impulse_responses',
        split='train', streaming=True
    )
    count = 0
    for row in tqdm(rir_dataset):
        name = row['audio']['path'].split('/')[-1]
        wav.write(os.path.join('mit_rirs', name), 16000,
                  (row['audio']['array'] * 32767).astype(np.int16))
        count += 1
    print(f'[OK] MIT RIRs: {count} arquivos')
else:
    n = len([f for f in os.listdir('mit_rirs') if f.endswith('.wav')])
    print(f'[SKIP] MIT RIRs já existem ({n} arquivos)')

# ============================================================
# 3b. AudioSet background noise
# Fonte correta: agkphysics/AudioSet (tar files)
# ============================================================
if not os.path.exists('audioset_16k') or len(os.listdir('audioset_16k')) == 0:
    print('\n[3b] Baixando AudioSet...')
    os.makedirs('audioset', exist_ok=True)
    os.makedirs('audioset_16k', exist_ok=True)
    fname = 'bal_train09.tar'
    link = f'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}'
    !wget -q -O audioset/{fname} {link}
    !cd audioset && tar -xf {fname}
    from pathlib import Path
    flac_files = list(Path('audioset/audio').glob('**/*.flac'))
    print(f'  Convertendo {len(flac_files)} flac → wav 16kHz...')
    audioset_dataset = datasets.Dataset.from_dict({'audio': [str(i) for i in flac_files]})
    audioset_dataset = audioset_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000))
    for row in tqdm(audioset_dataset):
        name = row['audio']['path'].split('/')[-1].replace('.flac', '.wav')
        wav.write(os.path.join('audioset_16k', name), 16000,
                  (row['audio']['array'] * 32767).astype(np.int16))
    print(f'[OK] AudioSet: {len(os.listdir("audioset_16k"))} clips')
else:
    print(f'[SKIP] AudioSet já existe ({len(os.listdir("audioset_16k"))} clips)')

# ============================================================
# 3c. FMA (música) – 1 hora
# Fonte correta: rudraml/fma (name="small", split="train")
# ============================================================
if not os.path.exists('fma') or len(os.listdir('fma')) == 0:
    print('\n[3c] Baixando FMA música (1 hora)...')
    os.makedirs('fma', exist_ok=True)
    fma_dataset = datasets.load_dataset('rudraml/fma', name='small', split='train', streaming=True)
    fma_dataset = iter(fma_dataset.cast_column('audio', datasets.Audio(sampling_rate=16000)))
    n_hours = 1
    count = 0
    for i in tqdm(range(n_hours * 3600 // 30)):
        row = next(fma_dataset)
        name = row['audio']['path'].split('/')[-1].replace('.mp3', '.wav')
        wav.write(os.path.join('fma', name), 16000,
                  (row['audio']['array'] * 32767).astype(np.int16))
        count += 1
    print(f'[OK] FMA: {count} clips')
else:
    print(f'[SKIP] FMA já existe ({len(os.listdir("fma"))} clips)')

# ============================================================
# 3d. ACAV100M features e Validation features
# Fonte correta: wget direto (arquivos .npy)
# ============================================================
hf_features = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main'

if not os.path.exists('openwakeword_features_ACAV100M_2000_hrs_16bit.npy'):
    print('\n[3d] Baixando ACAV100M features (~2 GB, pode demorar)...')
    !wget -q --show-progress '{hf_features}/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    print('[OK] ACAV100M features')
else:
    print('[SKIP] ACAV100M features já existe')

if not os.path.exists('validation_set_features.npy'):
    print('\n[3e] Baixando validation features...')
    !wget -q --show-progress '{hf_features}/validation_set_features.npy'
    print('[OK] Validation features')
else:
    print('[SKIP] Validation features já existe')

# Resumo
print()
print('Resumo dos dados baixados:')
checks = [
    ('mit_rirs', '*.wav', 'MIT RIRs'),
    ('audioset_16k', '*.wav', 'AudioSet'),
    ('fma', '*.wav', 'FMA'),
]
for d, pat, label in checks:
    if os.path.exists(d):
        n = len([f for f in os.listdir(d) if f.endswith('.wav')])
        print(f'  {label}: {n} arquivos')
    else:
        print(f'  {label}: FALTANDO!')
for f, label in [('openwakeword_features_ACAV100M_2000_hrs_16bit.npy','ACAV100M'),
                  ('validation_set_features.npy','Validation')]:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / 1024 / 1024
        print(f'  {label}: {size_mb:.1f} MB')
    else:
        print(f'  {label}: FALTANDO!')

print()
print('[OK] ETAPA 3 CONCLUÍDA!')

## Etapa 4: Gerar Clips de Treinamento
Gera ~24.000 clips com vozes pt_BR. **Tempo: ~20-40 min**

In [ ]:
import os, subprocess, uuid, random

target_word = 'letícia'
model_name  = 'leticia'
n_positive  = 10000
n_val       = 2000

ptbr_voices   = ['piper_voices_ptbr/pt_BR-faber-medium.onnx',
                  'piper_voices_ptbr/pt_BR-edresson-low.onnx']
length_scales = [0.8,0.85,0.9,0.95,1.0,1.05,1.1,1.15,1.2,1.25]
noise_scales  = [0.5,0.6,0.667,0.7,0.8,0.9,0.98]
noise_ws      = [0.5,0.6,0.7,0.8,0.9,0.98]

negative_words = [
    'patrícia','notícia','delícia','justiça','preguiça',
    'milícia','malícia','polícia','carência','urgência',
    'letivo','letrada','legítima','legião','elétrica',
    'lícia','alícia','felícia','luciana','larissa',
    'olá','bom dia','boa noite','obrigado','por favor',
    'ligar','desligar','acender','apagar','aumentar',
    'diminuir','temperatura','música','que horas são',
    'televisão','computador','celular','internet','cozinha',
]

base = f'./my_custom_model/{model_name}'
dirs = {
    'positive_train': f'{base}/positive_train',
    'positive_test':  f'{base}/positive_test',
    'negative_train': f'{base}/negative_train',
    'negative_test':  f'{base}/negative_test',
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

def gen_clips(text, out_dir, n, label):
    existing = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    if existing >= int(n * 0.95):
        print(f'  [SKIP] {label}: {existing} clips')
        return
    needed = n - existing
    texts  = [text] if isinstance(text, str) else text
    count  = 0
    while count < needed:
        for voice in ptbr_voices:
            for word in texts:
                out = os.path.join(out_dir, f'{uuid.uuid4().hex}.wav')
                try:
                    subprocess.run(
                        ['piper','--model',voice,'--output_file',out,
                         '--length-scale',str(random.choice(length_scales)),
                         '--noise-scale', str(random.choice(noise_scales)),
                         '--noise-w',     str(random.choice(noise_ws))],
                        input=word, capture_output=True, text=True, timeout=30
                    )
                    if os.path.exists(out): count += 1
                except Exception: pass
                if count >= needed: break
            if count >= needed: break
        if count % 500 == 0 and count > 0:
            print(f'  {label}: {count}/{needed}...')
    total = len([f for f in os.listdir(out_dir) if f.endswith('.wav')])
    print(f'  [OK] {label}: {total} clips')

print('[4a] Positivos treino...'); gen_clips(target_word, dirs['positive_train'], n_positive, 'Positivos treino')
print('[4b] Positivos val...');    gen_clips(target_word, dirs['positive_test'],  n_val,      'Positivos val')
print('[4c] Negativos treino...'); gen_clips(negative_words, dirs['negative_train'], n_positive, 'Negativos treino')
print('[4d] Negativos val...');    gen_clips(negative_words, dirs['negative_test'],  n_val,      'Negativos val')

print()
for label, d in dirs.items():
    n = len([f for f in os.listdir(d) if f.endswith('.wav')])
    print(f'  {label}: {n} clips')
print('[OK] ETAPA 4 CONCLUÍDA!')

## Etapa 5: Augmentação e Extração de Features
**Tempo: ~15-30 min**

In [ ]:
import os, sys, yaml

model_name = 'leticia'

# Determinar diretório de RIR
rir_dir = 'mit_rirs' if os.path.exists('mit_rirs') and \
          len([f for f in os.listdir('mit_rirs') if f.endswith('.wav')]) > 0 \
          else 'piper-sample-generator/impulses'
print(f'RIR dir: {os.path.abspath(rir_dir)}')

config = {
    'model_name': model_name,
    'target_phrase': ['letícia'],
    'custom_negative_phrases': ['patrícia','notícia','delícia','justiça','milícia','malícia','polícia','alícia','felícia'],
    'n_samples': 10000,
    'n_samples_val': 2000,
    'tts_batch_size': 50,
    'augmentation_batch_size': 16,
    'piper_sample_generator_path': os.path.abspath('./piper-sample-generator'),
    'output_dir': os.path.abspath('./my_custom_model'),
    'rir_paths': [os.path.abspath(rir_dir)],
    'background_paths': [os.path.abspath('./audioset_16k'), os.path.abspath('./fma')],
    'background_paths_duplication_rate': [1, 1],
    'augmentation_rounds': 1,
    'false_positive_validation_data_path': os.path.abspath('./validation_set_features.npy'),
    'feature_data_files': {'ACAV100M_sample': os.path.abspath('./openwakeword_features_ACAV100M_2000_hrs_16bit.npy')},
    'batch_n_per_class': {'ACAV100M_sample': 1024, 'adversarial_negative': 50, 'positive': 50},
    'model_type': 'dnn',
    'layer_size': 32,
    'steps': 50000,
    'max_negative_weight': 1500,
    'target_false_positives_per_hour': 0.2,
}

with open('my_model.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print('Config salva em my_model.yaml')

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips
print('[OK] ETAPA 5 CONCLUÍDA!')

## Etapa 6: Treinar o Modelo
**Tempo: ~30-60 min com GPU T4**

In [ ]:
import sys
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model --convert_to_tflite
print('[OK] ETAPA 6 CONCLUÍDA!')

## Etapa 7: Verificar e Baixar

In [ ]:
import os, glob
from google.colab import files

model_name  = 'leticia'
onnx_path   = f'my_custom_model/{model_name}.onnx'
tflite_path = f'my_custom_model/{model_name}.tflite'

# Fallback: converter se TFLite não foi gerado
if os.path.exists(onnx_path) and not os.path.exists(tflite_path):
    print('Convertendo ONNX → TFLite...')
    !onnx2tf -i {onnx_path} -o my_custom_model/tf_model -oiqt
    for f in glob.glob('my_custom_model/tf_model/**/*.tflite', recursive=True):
        import shutil; shutil.copy2(f, tflite_path)
        break

print('\nModelos gerados:')
for path, label in [(onnx_path,'ONNX'), (tflite_path,'TFLite')]:
    if os.path.exists(path):
        print(f'  ✅ {label}: {os.path.getsize(path)/1024:.1f} KB')
    else:
        print(f'  ❌ {label}: não encontrado')

for path in [tflite_path, onnx_path]:
    if os.path.exists(path):
        files.download(path)

print()
print('🎉 CONCLUÍDO!')
print('  Copie leticia.tflite para /share/openwakeword/ no HA Yellow')
print('  Reinicie o add-on openWakeWord e configure o pipeline')